# Feature Engineering Training v4 — SVM pada RGB CNN Baru

Gunakan kombinasi A/B dari **Feature Engineering Combination v4**. RGB memakai CNN empat tahap baru; AVG/NTSC tetap berasal dari model v1. Eksperimen ini membandingkan `cnn3` versus `all7`, SVM RBF, serta ensemble RBF+LinearSVC tanpa melatih CNN. Pemilihan tetap berdasarkan 40.000 training dan 10.000 validation. Baseline v2 yang perlu dilampaui adalah **93,90% validation**. Official test **tidak digunakan** di notebook ini. Semua artefak v1/v2/v3 aman karena output baru berada di v4.

## 1. Pustaka, konfigurasi, dan direktori

Jalankan dari `tugas-7` menggunakan kernel `envGPU`. Input dan output v4 berada di direktori baru yang tidak mengubah artefak v1/v2/v3. Parameter `MAX_SCREEN` dan `MAX_FULL` membatasi jumlah konfigurasi; tahap GridSearchCV CPU dapat membutuhkan waktu cukup lama.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import time

import joblib
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.svm import SVC, LinearSVC

SEED = 30092026
ROOT = Path.cwd().resolve()
if not (ROOT / "outputs" / "cifar10_7fitur_v4").is_dir() and (ROOT / "tugas-7" / "outputs" / "cifar10_7fitur_v4").is_dir():
    ROOT = ROOT / "tugas-7"
SOURCE_INPUT = ROOT / "outputs" / "cifar10_7fitur_v4" / "fusion"
V4 = ROOT / "outputs" / "cifar10_7fitur_v4" / "fusion"
for folder in ("model", "laporan"):
    (V4 / folder).mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = SOURCE_INPUT / "laporan" / "combination_manifest.json"
LEDGER_PATH = V4 / "laporan" / "search_ledger.json"
MAX_SCREEN = 16               # 2 representasi × 2 kelompok × 2 normalisasi × 2 kernel
MAX_FULL = 4                  # Kandidat terbaik tiap kombinasi representasi/kelompok
RUN_HYBRID = True             # RBF + LinearSVC; tidak melatih CNN
RUN_LATE_FUSION = False        # Tiga SVM per domain; lebih berat pada CPU
FINAL_FIT_POLICY = "only_if_beats_v2"  # Alternatif: "always" untuk eksperimen tambahan
BASELINE_V2_VALIDATION = 0.9390
print("Akar proyek:", ROOT)
print("Input kombinasi v4:", SOURCE_INPUT)
print("Output v4:", V4)


Akar proyek: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts
Input kombinasi v4: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v4/fusion
Output v4: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v4/fusion


## 2. Fungsi hash dan penyimpanan atomik

Hash menjaga agar hasil pencarian lama tidak tergabung dengan archive yang berubah. Ledger menyimpan parameter dan skor setelah setiap kandidat; tidak menyimpan data test.

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_object(obj):
    raw = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def atomic_json(path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

def atomic_joblib(path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    joblib.dump(payload, tmp, compress=3)
    os.replace(tmp, path)


## 3. Validasi arsip kandidat A dan B

**RGB** berasal dari GlobalAveragePooling2D 512-D CNN empat tahap yang baru dilatih. **AVG/NTSC** memakai Dense embedding 512-D model lama yang tetap dibekukan. A menggunakan original dan B menggunakan rata-rata original + flip pada masing-masing domain. Arsip v4 memuat tujuh kelompok; ablasi `cnn3` memakai hanya tiga blok awal. Periksa SHA, label, indeks, split, serta identitas checkpoint.

In [3]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest v4 tidak ditemukan: {MANIFEST_PATH}. Jalankan Combination v4 terlebih dahulu.")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if (manifest.get("run_id") != "cifar10_7fitur_v4" or manifest.get("seed") != SEED
        or manifest.get("block_names") != ["rgb", "avg", "ntsc", "hog", "lbp", "hsv", "rgb_stats"]):
    raise ValueError("Manifest v4 tidak sesuai skema atau seed")
archive_paths = {rep: SOURCE_INPUT / "fitur" / f"train_combined_{rep}.npz" for rep in ("A", "B")}
for rep, path in archive_paths.items():
    if not path.is_file() or sha256_file(path) != manifest["archive_sha256"][rep]:
        raise ValueError(f"Arsip {rep} hilang atau hash berbeda: {path}")
for variant, expected in zip(("rgb", "avg", "ntsc"), manifest["model_hashes"]):
    ckpt = ROOT / manifest["rgb_model_path"] if variant == "rgb" else ROOT / manifest[f"{variant}_model_path"]
    if not ckpt.is_file() or sha256_file(ckpt) != expected:
        raise ValueError(f"Checkpoint {variant} tidak cocok dengan provenance v4")

features = {}
for rep, path in archive_paths.items():
    with np.load(path, allow_pickle=False) as a:
        X = a["features"].astype(np.float32)
        if (X.shape != (50000, 2173) or not np.isfinite(X).all()
                or not np.array_equal(a["boundaries"], np.array([0, 512, 1024, 1536, 1860, 2116, 2164, 2173]))
                or str(a["split_hash"]) != manifest["split_hash"]
                or str(a["representation"]) != rep
                or not np.array_equal(a["indices"], np.arange(50000))
                or not np.array_equal(a["model_hashes"], manifest["model_hashes"])):
            raise ValueError(f"Dimensi, provenance, atau indeks kandidat {rep} salah")
        if rep == "A":
            y = a["y"].astype(np.int64)
            idx_train = a["train_idx"].astype(np.int64)
            idx_val = a["val_idx"].astype(np.int64)
            if (len(idx_train) != 40000 or len(idx_val) != 10000
                    or len(np.intersect1d(idx_train, idx_val))
                    or not np.array_equal(np.sort(np.r_[idx_train, idx_val]), np.arange(50000))):
                raise ValueError("Split 40K/10K tidak valid")
        elif (not np.array_equal(a["y"], y)
              or not np.array_equal(a["train_idx"], idx_train)
              or not np.array_equal(a["val_idx"], idx_val)):
            raise ValueError("Kandidat A/B tidak sejajar")
        features[rep] = X
print("Kandidat A/B valid:", {k: v.shape for k,v in features.items()})
print("Development split:", len(idx_train), len(idx_val), "| model hashes valid")


Kandidat A/B valid: {'A': (50000, 2173), 'B': (50000, 2173)}
Development split: 40000 10000 | model hashes valid


## 4. Transformer preprocessing berbasis kelompok

Dua mode dibandingkan: `block_l2=True` menggunakan StandardScaler dan L2 per kelompok sebelum pembobotan, sedangkan `False` menggunakan StandardScaler dan bobot lalu L2 gabungan. Statistik scaler hanya di-fit pada subset training di setiap evaluasi. Ketiga CNN selalu berada pada posisi tiga blok pertama.

In [4]:
class GroupPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, boundaries, weights, block_l2=False):
        self.boundaries = boundaries
        self.weights = weights
        self.block_l2 = block_l2

    def fit(self, X, y=None):
        self.boundaries_ = tuple(int(i) for i in self.boundaries)
        if self.boundaries_[0] != 0 or self.boundaries_[-1] != X.shape[1]:
            raise ValueError("Batas fitur tidak sesuai")
        if len(self.weights) != len(self.boundaries_) - 1:
            raise ValueError("Bobot kelompok tidak sesuai")
        self.scalers_ = []
        for left, right in zip(self.boundaries_[:-1], self.boundaries_[1:]):
            self.scalers_.append(StandardScaler().fit(X[:, left:right]))
        return self

    def transform(self, X):
        if X.shape[1] != self.boundaries_[-1]:
            raise ValueError("Dimensi transform salah")
        groups = []
        for scaler, w, left, right in zip(self.scalers_, self.weights,
                                           self.boundaries_[:-1], self.boundaries_[1:]):
            block = scaler.transform(X[:, left:right]).astype(np.float32)
            if self.block_l2:
                block = Normalizer().transform(block)
            groups.append(block * np.float32(w))
        return Normalizer().transform(np.concatenate(groups, axis=1)).astype(np.float32)

def make_spec(rep, group_set, block_l2, C, gamma):
    if group_set == "cnn3":
        boundaries = (0, 512, 1024, 1536)
        weights = (2.0, 1.0, 1.0) if block_l2 else (1.0, 0.7, 0.7)
    elif group_set == "all7":
        boundaries = (0, 512, 1024, 1536, 1860, 2116, 2164, 2173)
        weights = ((2., 1., 1., .1, .1, .1, .1) if block_l2
                   else (1., .7, .7, .1, .1, .1, .1))
    else:
        raise ValueError("Pilihan kelompok tidak dikenal")
    return {"rep": rep, "groups": group_set, "block_l2": bool(block_l2),
            "weights": weights, "boundaries": boundaries,
            "C": float(C), "gamma": gamma}

def matrix_for(spec):
    X = features[spec["rep"]]
    return X[:, :1536] if spec["groups"] == "cnn3" else X

def make_pipeline(spec):
    return Pipeline([
        ("groups", GroupPreprocessor(tuple(spec["boundaries"]), tuple(spec["weights"]), spec["block_l2"])),
        ("svc", SVC(C=spec["C"], gamma=spec["gamma"], kernel="rbf",
                    decision_function_shape="ovr", break_ties=True,
                    cache_size=1024, random_state=SEED))
    ])


## 5. Screening stratified dan identitas eksperimen

Screening memakai 12.000 sampel dari development training serta 3.000 dari development validation. PredefinedSplit memastikan bagian validasi tidak digunakan untuk `fit` scaler atau SVM. Fingerprint mengikat hasil pada archive, source split, dan resep screening.

In [5]:
idx_screen_train, _ = train_test_split(idx_train, train_size=12000,
                                      random_state=SEED, stratify=y[idx_train])
idx_screen_val, _ = train_test_split(idx_val, train_size=3000,
                                    random_state=SEED, stratify=y[idx_val])
IDENTITY = {
    "run": "cifar10_7fitur_v4", "seed": SEED,
    "v4_manifest_sha256": sha256_file(MANIFEST_PATH),
    "source_archives": {k: sha256_file(v) for k,v in archive_paths.items()},
    "split_hash": manifest["split_hash"],
    "screen_train_sha256": hashlib.sha256(idx_screen_train.tobytes()).hexdigest(),
    "screen_val_sha256": hashlib.sha256(idx_screen_val.tobytes()).hexdigest(),
    "design": "v4-rgb-512gap-avgntsc-v1-ablations-hybrid-1"
}
IDENTITY_SHA = sha256_object(IDENTITY)
if LEDGER_PATH.exists():
    ledger = json.loads(LEDGER_PATH.read_text(encoding="utf-8"))
    if ledger.get("identity_sha256") != IDENTITY_SHA:
        raise ValueError("Ledger lama tidak sesuai archive/resep. Jangan gabungkan eksperimen.")
else:
    ledger = {"identity_sha256": IDENTITY_SHA, "identity": IDENTITY,
              "screen": {}, "full": {}, "ensemble": {}, "late": {},
              "created_utc": datetime.now(timezone.utc).isoformat()}
    atomic_json(LEDGER_PATH, ledger)
print("Identity:", IDENTITY_SHA[:16], "| Kandidat yang tersimpan:",
      {key: len(ledger[key]) for key in ("screen", "full", "ensemble", "late")})


Identity: 19e41a54de232ff8 | Kandidat yang tersimpan: {'screen': 0, 'full': 0, 'ensemble': 0, 'late': 0}


## 6. GridSearchCV screening dengan checkpoint

Setiap pemanggilan `GridSearchCV` mengevaluasi satu konfigurasi, sehingga hasil disimpan setelah selesai. Konfigurasi mencakup representation A/B, tiga atau tujuh kelompok, normalisasi per kelompok aktif/nonaktif, dan dua pasangan parameter RBF. Nilai validasi screening **bukan** dasar akhir pemilihan model.

In [6]:
candidates = [make_spec(rep, groups, block_l2, C, gamma)
              for rep in ("B", "A")
              for groups in ("cnn3", "all7")
              for block_l2 in (False, True)
              for C, gamma in ((1, "scale"), (10, 0.5))]
candidates = candidates[:MAX_SCREEN]

def spec_key(spec):
    return sha256_object(spec)[:20]

for number, spec in enumerate(candidates, 1):
    key = spec_key(spec)
    if key in ledger["screen"]:
        print(f"Screen {number}/{len(candidates)}: cache {key} = {ledger['screen'][key]['accuracy']:.4f}")
        continue
    X = matrix_for(spec)
    X_screen = np.concatenate((X[idx_screen_train], X[idx_screen_val]), axis=0)
    y_screen = np.r_[y[idx_screen_train], y[idx_screen_val]]
    predefined = PredefinedSplit(np.r_[np.full(len(idx_screen_train), -1),
                                      np.zeros(len(idx_screen_val), dtype=int)])
    grid = GridSearchCV(make_pipeline(spec),
                        {"svc__C": [spec["C"]], "svc__gamma": [spec["gamma"]]},
                        scoring="accuracy", cv=predefined, refit=False,
                        n_jobs=1, verbose=0, error_score="raise")
    start = time.perf_counter()
    grid.fit(X_screen, y_screen)
    seconds = time.perf_counter() - start
    ledger["screen"][key] = {"spec": spec, "accuracy": float(grid.best_score_),
                             "seconds": seconds, "n_train": len(idx_screen_train),
                             "n_val": len(idx_screen_val)}
    atomic_json(LEDGER_PATH, ledger)
    print(f"Screen {number}/{len(candidates)} {spec['rep']}-{spec['groups']} "
          f"L2={spec['block_l2']} C={spec['C']} gamma={spec['gamma']} "
          f"acc={grid.best_score_:.4%} waktu={seconds/60:.1f} menit", flush=True)
print("Screening selesai. Kandidat tersimpan:", len(ledger["screen"]))


Screen 1/16 B-cnn3 L2=False C=1.0 gamma=scale acc=94.3667% waktu=0.1 menit
Screen 2/16 B-cnn3 L2=False C=10.0 gamma=0.5 acc=94.5000% waktu=0.1 menit
Screen 3/16 B-cnn3 L2=True C=1.0 gamma=scale acc=93.9000% waktu=0.1 menit
Screen 4/16 B-cnn3 L2=True C=10.0 gamma=0.5 acc=93.9000% waktu=0.1 menit
Screen 5/16 B-all7 L2=False C=1.0 gamma=scale acc=94.4000% waktu=0.2 menit
Screen 6/16 B-all7 L2=False C=10.0 gamma=0.5 acc=94.3667% waktu=0.1 menit
Screen 7/16 B-all7 L2=True C=1.0 gamma=scale acc=93.8667% waktu=0.2 menit
Screen 8/16 B-all7 L2=True C=10.0 gamma=0.5 acc=93.9000% waktu=0.1 menit
Screen 9/16 A-cnn3 L2=False C=1.0 gamma=scale acc=94.3667% waktu=0.1 menit
Screen 10/16 A-cnn3 L2=False C=10.0 gamma=0.5 acc=94.5000% waktu=0.1 menit
Screen 11/16 A-cnn3 L2=True C=1.0 gamma=scale acc=94.0667% waktu=0.1 menit
Screen 12/16 A-cnn3 L2=True C=10.0 gamma=0.5 acc=94.1000% waktu=0.1 menit
Screen 13/16 A-all7 L2=False C=1.0 gamma=scale acc=94.3667% waktu=0.2 menit
Screen 14/16 A-all7 L2=False C=10

## 7. Pilih kandidat untuk validasi penuh

Dari screening dipilih kandidat terbaik untuk masing-masing dari empat kombinasi A/B × cnn3/all7. Ini mencegah satu kelompok mendominasi seluruh pengujian penuh. Seluruh skor final tetap dihitung pada 10.000 development validation.

In [7]:
if not ledger["screen"]:
    raise RuntimeError("Belum ada hasil screening yang dapat dievaluasi")
full_candidates = []
for rep in ("B", "A"):
    for groups in ("cnn3", "all7"):
        choices = list(v for v in ledger["screen"].values()
                       if v["spec"]["rep"] == rep and v["spec"]["groups"] == groups)
        if choices:
            full_candidates.append(max(choices, key=lambda v: v["accuracy"])["spec"])
full_candidates = full_candidates[:MAX_FULL]
for i, spec in enumerate(full_candidates, 1):
    print(i, spec)


1 {'rep': 'B', 'groups': 'cnn3', 'block_l2': False, 'weights': (1.0, 0.7, 0.7), 'boundaries': (0, 512, 1024, 1536), 'C': 10.0, 'gamma': 0.5}
2 {'rep': 'B', 'groups': 'all7', 'block_l2': False, 'weights': (1.0, 0.7, 0.7, 0.1, 0.1, 0.1, 0.1), 'boundaries': (0, 512, 1024, 1536, 1860, 2116, 2164, 2173), 'C': 1.0, 'gamma': 'scale'}
3 {'rep': 'A', 'groups': 'cnn3', 'block_l2': False, 'weights': (1.0, 0.7, 0.7), 'boundaries': (0, 512, 1024, 1536), 'C': 10.0, 'gamma': 0.5}
4 {'rep': 'A', 'groups': 'all7', 'block_l2': False, 'weights': (1.0, 0.7, 0.7, 0.1, 0.1, 0.1, 0.1), 'boundaries': (0, 512, 1024, 1536, 1860, 2116, 2164, 2173), 'C': 10.0, 'gamma': 0.5}


## 8. Validasi penuh 40K/10K dan cache model terbaik

Untuk setiap kandidat, semua transformer serta RBF SVM di-fit pada 40.000 sampel training, kemudian dievaluasi pada 10.000 validation. Model development terbaik disimpan agar eksperimen ensemble tidak perlu selalu mengulang fitting RBF. Model development **bukan** model final 50K.

In [8]:
best_dev_path = V4 / "model" / "best_development_rbf.joblib"
for i, spec in enumerate(full_candidates, 1):
    key = spec_key(spec)
    if key in ledger["full"]:
        print(f"Full {i}/{len(full_candidates)}: cache {key} = {ledger['full'][key]['accuracy']:.4%}")
        continue
    X = matrix_for(spec)
    estimator = make_pipeline(spec)
    start = time.perf_counter()
    estimator.fit(X[idx_train], y[idx_train])
    predicted = estimator.predict(X[idx_val])
    accuracy = float(accuracy_score(y[idx_val], predicted))
    seconds = time.perf_counter() - start
    ledger["full"][key] = {"spec": spec, "accuracy": accuracy,
                           "seconds": seconds, "correct": int((predicted == y[idx_val]).sum()),
                           "n_train": len(idx_train), "n_val": len(idx_val)}
    atomic_json(LEDGER_PATH, ledger)
    best_key = max(ledger["full"], key=lambda k: ledger["full"][k]["accuracy"])
    if best_key == key:
        atomic_joblib(best_dev_path, {"identity_sha256": IDENTITY_SHA,
                                      "spec_key": key, "pipeline": estimator})
    print(f"Full {i}/{len(full_candidates)} {spec['rep']}-{spec['groups']}: "
          f"{accuracy:.4%} ({seconds/60:.1f} menit)", flush=True)
if not ledger["full"]:
    raise RuntimeError("Belum ada kandidat tervalidasi penuh")
best_rbf_key = max(ledger["full"], key=lambda k: ledger["full"][k]["accuracy"])
best_rbf = ledger["full"][best_rbf_key]
print("RBF development terbaik:", best_rbf["accuracy"], best_rbf["spec"])


Full 1/4 B-cnn3: 94.5300% (0.3 menit)
Full 2/4 B-all7: 94.6000% (0.7 menit)
Full 3/4 A-cnn3: 94.0900% (0.3 menit)
Full 4/4 A-all7: 94.1400% (0.5 menit)
RBF development terbaik: 0.946 {'rep': 'B', 'groups': 'all7', 'block_l2': False, 'weights': (1.0, 0.7, 0.7, 0.1, 0.1, 0.1, 0.1), 'boundaries': (0, 512, 1024, 1536, 1860, 2116, 2164, 2173), 'C': 1.0, 'gamma': 'scale'}


## 9. Fungsi normalisasi decision scores

SVC dan LinearSVC menghasilkan decision scores dengan skala berbeda. Statistik rerata dan simpangan baku untuk setiap kelas dihitung hanya dari **training**, bukan validation. Ensemble memakai bobot yang dipilih menggunakan validation; tidak ada koreksi per indeks gambar.

In [9]:
def score_stats(scores):
    scores = np.asarray(scores, dtype=np.float64)
    return {"mean": scores.mean(axis=0),
            "std": np.maximum(scores.std(axis=0), 1e-6)}

def score_transform(scores, stats):
    return (np.asarray(scores, dtype=np.float64) - stats["mean"]) / stats["std"]

def normalize_stats_for_save(stats):
    return {"mean": np.asarray(stats["mean"]), "std": np.asarray(stats["std"])}


## 10. Ensemble RBF + LinearSVC pada development

Latih satu LinearSVC pada representasi yang sama dengan RBF development terbaik. Uji bobot pencampuran di validation, dengan skor kedua model distandardisasi berdasarkan distribusi training. Simpan hasil dan parameter agar dapat dibandingkan dengan RBF tunggal. Langkah ini tidak menyentuh label official test.

In [10]:
if RUN_HYBRID:
    if "hybrid" in ledger["ensemble"]:
        if ledger["ensemble"]["hybrid"].get("source_rbf_key") != best_rbf_key:
            raise ValueError("Ensemble lama berasal dari kandidat RBF yang berbeda")
        print("Ensemble development telah tersimpan:", ledger["ensemble"]["hybrid"]["accuracy"])
    else:
        spec = best_rbf["spec"]
        X = matrix_for(spec)
        stored = joblib.load(best_dev_path) if best_dev_path.exists() else None
        if stored is not None and (stored.get("identity_sha256") != IDENTITY_SHA or
                                    stored.get("spec_key") != best_rbf_key):
            stored = None
        if stored is None:
            estimator = make_pipeline(spec).fit(X[idx_train], y[idx_train])
        else:
            estimator = stored["pipeline"]
        prep = estimator.named_steps["groups"]
        rbf = estimator.named_steps["svc"]
        X_tr_p = prep.transform(X[idx_train])
        X_val_p = prep.transform(X[idx_val])
        start = time.perf_counter()
        linear = LinearSVC(C=1.0, dual=False, max_iter=5000, random_state=SEED)
        linear.fit(X_tr_p, y[idx_train])
        if not np.array_equal(rbf.classes_, linear.classes_) or not np.array_equal(rbf.classes_, np.arange(10)):
            raise ValueError("Urutan kelas RBF dan LinearSVC tidak identik")
        rbf_stats = score_stats(rbf.decision_function(X_tr_p))
        lin_stats = score_stats(linear.decision_function(X_tr_p))
        rbf_val = score_transform(rbf.decision_function(X_val_p), rbf_stats)
        lin_val = score_transform(linear.decision_function(X_val_p), lin_stats)
        results = {}
        for linear_weight in (0.25, 0.5, 0.75, 1.0):
            pred = np.argmax(rbf_val + linear_weight * lin_val, axis=1)
            results[str(linear_weight)] = {"accuracy": float(accuracy_score(y[idx_val], pred)),
                                            "correct": int((y[idx_val] == pred).sum())}
        best_weight = max(results, key=lambda w: results[w]["accuracy"])
        ledger["ensemble"]["hybrid"] = {
            "spec": spec, "source_rbf_key": best_rbf_key,
            "accuracy": results[best_weight]["accuracy"],
            "correct": results[best_weight]["correct"], "linear_weight": float(best_weight),
            "all_weights": results, "seconds": time.perf_counter()-start,
            "n_train": len(idx_train), "n_val": len(idx_val)
        }
        atomic_json(LEDGER_PATH, ledger)
        print("Ensemble terbaik:", ledger["ensemble"]["hybrid"])
else:
    print("Ensemble dilewati sesuai RUN_HYBRID=False")


Ensemble terbaik: {'spec': {'rep': 'B', 'groups': 'all7', 'block_l2': False, 'weights': (1.0, 0.7, 0.7, 0.1, 0.1, 0.1, 0.1), 'boundaries': (0, 512, 1024, 1536, 1860, 2116, 2164, 2173), 'C': 1.0, 'gamma': 'scale'}, 'source_rbf_key': '2d7605cdeb2668a0d098', 'accuracy': 0.9454, 'correct': 9454, 'linear_weight': 0.5, 'all_weights': {'0.25': {'accuracy': 0.9442, 'correct': 9442}, '0.5': {'accuracy': 0.9454, 'correct': 9454}, '0.75': {'accuracy': 0.9454, 'correct': 9454}, '1.0': {'accuracy': 0.9453, 'correct': 9453}}, 'seconds': 88.30106291000266, 'n_train': 40000, 'n_val': 10000}


## 11. Late fusion dari tiga SVM per domain

Setiap representasi RGB, AVG, dan NTSC diberi StandardScaler serta normalisasi L2 sendiri dan dilatih memakai RBF SVM. Prediksi dipadukan pada tingkat `decision_function`, bukan concatenation. Statistik skor dipelajari dari data training. Weighting diuji pada validation. Kode ini lebih mahal karena membutuhkan tiga fitting SVM development; bisa dilewati dengan `RUN_LATE_FUSION=False`.

In [11]:
if RUN_LATE_FUSION:
    late_rep = best_rbf["spec"]["rep"]
    late_models = {}
    start = time.perf_counter()
    for i, domain in enumerate(("rgb", "avg", "ntsc")):
        model_path = V4 / "model" / f"late_dev_{late_rep}_{domain}.joblib"
        if model_path.exists():
            domain_data = joblib.load(model_path)
            if domain_data.get("identity_sha256") != IDENTITY_SHA:
                raise ValueError(f"Cache late-fusion {domain} tidak kompatibel")
        else:
            X = features[late_rep][:, i*512:(i+1)*512]
            pipe = Pipeline([
                ("scaler", StandardScaler()), ("l2", Normalizer()),
                ("svc", SVC(kernel="rbf", C=3.0, gamma="scale",
                            break_ties=True, cache_size=1024,
                            decision_function_shape="ovr", random_state=SEED))
            ])
            pipe.fit(X[idx_train], y[idx_train])
            if not np.array_equal(pipe.named_steps["svc"].classes_, np.arange(10)):
                raise ValueError("Kelas late SVC tidak cocok")
            stats = score_stats(pipe.decision_function(X[idx_train]))
            domain_data = {"identity_sha256": IDENTITY_SHA, "pipeline": pipe,
                           "stats": normalize_stats_for_save(stats), "domain": domain,
                           "representation": late_rep}
            atomic_joblib(model_path, domain_data)
        late_models[domain] = domain_data
        print("Late development domain siap:", domain, flush=True)
    val_scores = []
    for i, domain in enumerate(("rgb", "avg", "ntsc")):
        X = features[late_rep][idx_val, i*512:(i+1)*512]
        data = late_models[domain]
        val_scores.append(score_transform(data["pipeline"].decision_function(X), data["stats"]))
    variants = {}
    for weights in ((1., 1., 1.), (1., .7, .7), (2., 1., 1.)):
        pred = np.argmax(sum(w*s for w,s in zip(weights,val_scores)), axis=1)
        variants[json.dumps(weights)] = {"accuracy": float(accuracy_score(y[idx_val], pred)),
                                         "correct": int((y[idx_val] == pred).sum())}
    winner = max(variants, key=lambda w: variants[w]["accuracy"])
    ledger["late"]["best"] = {"source_rbf_key": best_rbf_key,
                               "representation": late_rep, "weights": json.loads(winner),
                               "accuracy": variants[winner]["accuracy"],
                               "correct": variants[winner]["correct"],
                               "c": 3.0, "gamma": "scale", "all_weights": variants,
                               "seconds": time.perf_counter()-start,
                               "n_train": len(idx_train), "n_val": len(idx_val)}
    atomic_json(LEDGER_PATH, ledger)
    print("Late fusion terbaik:", ledger["late"]["best"])
else:
    print("Late fusion dilewati sesuai RUN_LATE_FUSION=False")


Late fusion dilewati sesuai RUN_LATE_FUSION=False


## 12. Bandingkan dan bekukan konfigurasi terbaik

Pemilihan hanya berdasarkan 10.000 development validation. **Baseline yang dipertahankan adalah v2 (93,90%)**, bukan nilai v4. Jika kandidat v4 tidak mengungguli v2 pada validation, tahap fitting final v4 dilewati dan seluruh artefak v2 tetap utuh. `selection.json` yang telah disimpan tidak boleh berubah dalam experiment ID yang sama.

In [12]:
choices = [{"kind": "rbf", "accuracy": best_rbf["accuracy"],
            "correct": best_rbf["correct"], "spec": best_rbf["spec"]}]
if RUN_HYBRID and "hybrid" in ledger["ensemble"]:
    e = ledger["ensemble"]["hybrid"]
    choices.append({"kind": "hybrid", "accuracy": e["accuracy"],
                    "correct": e["correct"], "spec": e["spec"],
                    "linear_weight": e["linear_weight"]})
if RUN_LATE_FUSION and "best" in ledger["late"]:
    e = ledger["late"]["best"]
    if e.get("source_rbf_key") != best_rbf_key:
        raise ValueError("Late fusion development berasal dari RBF source berbeda")
    choices.append({"kind": "late", "accuracy": e["accuracy"],
                    "correct": e["correct"], "representation": e["representation"],
                    "weights": e["weights"], "c": e["c"], "gamma": e["gamma"]})
for choice in choices:
    print(f"{choice['kind']:>8s} | validation {choice['accuracy']:.4%} | benar {choice['correct']}/10000")
selected = max(choices, key=lambda c: (c["accuracy"], -["rbf","hybrid","late"].index(c["kind"])))
selection = {"run_id": "cifar10_7fitur_v4", "identity_sha256": IDENTITY_SHA,
             "source_manifest_sha256": sha256_file(MANIFEST_PATH),
             "selected": selected, "v2_validation_accuracy": BASELINE_V2_VALIDATION,
             "validation_delta": selected["accuracy"] - BASELINE_V2_VALIDATION,
             "n_development_train": 40000, "n_development_validation": 10000,
             "model_source": "RGB v4 frozen pooling embeddings; AVG/NTSC v1 frozen Dense embeddings",
             "post_exposure_test": True}
selection = json.loads(json.dumps(selection))  # tuple -> list; stabil setelah JSON round-trip
selection_path = V4 / "laporan" / "selection.json"
if selection_path.exists():
    frozen = json.loads(selection_path.read_text(encoding="utf-8"))
    if frozen != selection:
        raise ValueError("selection.json sudah dikunci dengan konfigurasi berbeda. Gunakan run baru.")
else:
    atomic_json(selection_path, selection)
print("Seleksi final:", selected)
print("Dibanding v2:", f"{selection['validation_delta']:+.2%}")


     rbf | validation 94.6000% | benar 9460/10000
  hybrid | validation 94.5400% | benar 9454/10000
Seleksi final: {'kind': 'rbf', 'accuracy': 0.946, 'correct': 9460, 'spec': {'rep': 'B', 'groups': 'all7', 'block_l2': False, 'weights': (1.0, 0.7, 0.7, 0.1, 0.1, 0.1, 0.1), 'boundaries': (0, 512, 1024, 1536, 1860, 2116, 2164, 2173), 'C': 1.0, 'gamma': 'scale'}}
Dibanding v2: +0.70%


## 13. Final fitting 50K dan penyimpanan model v4

CNN RGB baru telah dilatih pada notebook RGB Training v4; notebook ini **hanya melatih ulang scaler dan classifier SVM**, menggunakan semua 50.000 fitur resmi training setelah konfigurasi dikunci. Dengan `FINAL_FIT_POLICY="only_if_beats_v2"`, fitting dilewati ketika skor validation v4 tidak lebih tinggi dari 93,90%. Hal tersebut tidak merupakan error. Official test belum diakses.

In [13]:
model_path = V4 / "model" / "fusion_v4_final.joblib"
should_fit = FINAL_FIT_POLICY == "always" or selected["accuracy"] > BASELINE_V2_VALIDATION
if FINAL_FIT_POLICY not in ("always", "only_if_beats_v2"):
    raise ValueError("FINAL_FIT_POLICY tidak dikenal")
if not should_fit:
    print("SKIP final fit: validasi v4 tidak melampaui baseline 93,90%. Artefak v2 tetap utuh.")
elif model_path.exists():
    old = joblib.load(model_path)
    if (old.get("selection_sha256") != sha256_file(selection_path)
            or old.get("identity_sha256") != IDENTITY_SHA or old.get("fit_count") != 50000):
        raise ValueError("Model final v4 yang sudah ada tidak sesuai seleksi")
    print("Model final v4 ditemukan dan cocok; tidak melakukan fitting ulang.")
else:
    start = time.perf_counter()
    if selected["kind"] in ("rbf", "hybrid"):
        spec = selected["spec"]
        X = matrix_for(spec)
        prep = GroupPreprocessor(tuple(spec["boundaries"]), tuple(spec["weights"]), spec["block_l2"])
        Z = prep.fit_transform(X)
        rbf = SVC(kernel="rbf", C=spec["C"], gamma=spec["gamma"],
                  break_ties=True, decision_function_shape="ovr",
                  cache_size=1024, random_state=SEED).fit(Z, y)
        payload = {"kind": selected["kind"], "spec": spec,
                   "preprocessor": prep, "rbf": rbf}
        if selected["kind"] == "hybrid":
            linear = LinearSVC(C=1.0, dual=False, max_iter=5000, random_state=SEED).fit(Z,y)
            payload.update({"linear": linear,
                            "rbf_stats": normalize_stats_for_save(score_stats(rbf.decision_function(Z))),
                            "linear_stats": normalize_stats_for_save(score_stats(linear.decision_function(Z))),
                            "linear_weight": selected["linear_weight"]})
    else:
        rep = selected["representation"]
        domains = []
        for i, domain in enumerate(("rgb","avg","ntsc")):
            X = features[rep][:, i*512:(i+1)*512]
            pipe = Pipeline([
                ("scaler", StandardScaler()), ("l2", Normalizer()),
                ("svc", SVC(kernel="rbf", C=selected["c"], gamma=selected["gamma"],
                            break_ties=True, decision_function_shape="ovr",
                            cache_size=1024, random_state=SEED))
            ]).fit(X,y)
            domains.append({"name": domain, "pipeline": pipe,
                            "stats": normalize_stats_for_save(score_stats(pipe.decision_function(X)))})
            print("Final late domain:", domain, flush=True)
        payload = {"kind": "late", "representation": rep,
                   "weights": selected["weights"], "domains": domains}
    payload.update({"run_id": "cifar10_7fitur_v4", "fit_count": 50000,
                    "selection_sha256": sha256_file(selection_path),
                    "identity_sha256": IDENTITY_SHA,
                    "source_archives": IDENTITY["source_archives"],
                    "fit_seconds": time.perf_counter() - start})
    atomic_joblib(model_path, payload)
    print("Model v4 disimpan:", model_path, "| waktu", round(payload["fit_seconds"]/60, 1), "menit")


Model v4 disimpan: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v4/fusion/model/fusion_v4_final.joblib | waktu 1.0 menit


## 14. Ringkasan eksperimen untuk laporan

Simpan ringkasan hasil lengkap, termasuk kandidat screening, full validation, ensemble, late fusion, dan alasan final fit dilewati apabila tidak ada peningkatan. Accuracy official test belum diketahui pada tahap ini.

In [14]:
report = {"run_id": "cifar10_7fitur_v4", "identity_sha256": IDENTITY_SHA,
          "screen_results": ledger["screen"], "full_results": ledger["full"],
          "ensemble_results": ledger["ensemble"], "late_results": ledger["late"],
          "selection": selection, "model_file_exists": model_path.exists(),
          "note": "official test tidak digunakan pada proses pemilihan"}
atomic_json(V4 / "laporan" / "development_report.json", report)
print("Ringkasan tersimpan. Validasi v4 terbaik:", f"{selected['accuracy']:.2%}")
print("Model final:", "tersedia" if model_path.exists() else "dilewati karena belum mengungguli v2")
print("Jalankan Testing v4 hanya setelah model final tersedia.")


Ringkasan tersimpan. Validasi v4 terbaik: 94.60%
Model final: tersedia
Jalankan Testing v4 hanya setelah model final tersedia.
